# Clasificador de triatominos — MobileNetV3 (Miranda)

## Instalar dependencias

In [ ]:
!pip install -q -U datasets

## Configuración

In [ ]:
import os, io, time, random, collections, requests
import numpy as np
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
GENEROS = ["Panstrongylus", "Rhodnius", "Triatoma"]
g2i = {g: i for i, g in enumerate(GENEROS)}
print("GPU:", tf.config.list_physical_devices("GPU"))

## Cargar datos (Miranda)

In [ ]:
from datasets import load_dataset

ds = load_dataset("Totan2305/triatominos-augmentado-parquet")["train"]
nombres = ds.features["label"].names
n = ds.num_rows
X = np.empty((n, 224, 224, 3), np.uint8)
y = np.empty(n, np.int64)
for i, ej in enumerate(ds):
    X[i] = np.asarray(ej["image"].convert("RGB").resize((224, 224)), np.uint8)
    y[i] = g2i[nombres[ej["label"]]]
print(len(X), collections.Counter(GENEROS[l] for l in y))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=SEED)

## Preparar datos y modelo

In [ ]:
AUTO = tf.data.AUTOTUNE

def make_ds(X, y, shuffle=False):
    def gen():
        for i in range(len(X)):
            yield X[i], y[i]
    d = tf.data.Dataset.from_generator(gen, output_signature=(
        tf.TensorSpec((224, 224, 3), tf.uint8), tf.TensorSpec((), tf.int64)))
    if shuffle:
        d = d.shuffle(4000, seed=SEED)
    d = d.map(lambda a, b: (tf.cast(a, tf.float32), b), num_parallel_calls=AUTO)
    return d.batch(32).prefetch(AUTO)

def construir():
    base = keras.applications.MobileNetV3Large(
        input_shape=(224, 224, 3), include_top=False,
        weights="imagenet", include_preprocessing=True)
    base.trainable = False
    model = keras.Sequential([
        keras.Input((224, 224, 3)),
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.2),
        layers.Dense(len(GENEROS), activation="softmax"),
    ])
    return model, base

## Entrenamiento

In [ ]:
train_ds = make_ds(X_train, y_train, shuffle=True)
test_ds = make_ds(X_test, y_test)
w = compute_class_weight("balanced", classes=np.arange(len(GENEROS)), y=y_train)
pesos = dict(enumerate(w))

model, base = construir()
model.compile(keras.optimizers.Adam(1e-3), "sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(train_ds, validation_data=test_ds, epochs=5, class_weight=pesos)

base.trainable = True
for l in base.layers:
    if isinstance(l, layers.BatchNormalization):
        l.trainable = False
model.compile(keras.optimizers.Adam(1e-5), "sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(train_ds, validation_data=test_ds, epochs=5, class_weight=pesos)

## Evaluación

In [ ]:
y_pred = model.predict(test_ds).argmax(1)
print(classification_report(y_test, y_pred, target_names=GENEROS, digits=4))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=GENEROS).plot(cmap="Blues", xticks_rotation=45)
plt.tight_layout(); plt.savefig("matriz_confusion.png", dpi=150); plt.show()

## Guardar modelo

In [ ]:
model.save("cnn_miranda.keras")
print("guardado:", "cnn_miranda.keras")